<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/ai-act-conformity/lessons/P01-L08-conformity-assessment-route/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/ai-act-conformity/lessons/P01-L08-conformity-assessment-route/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/ai-act-conformity/lessons/P01-L08-conformity-assessment-route/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/ai-act-conformity/lessons/P01-L08-conformity-assessment-route/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P01-L08 · The conformity assessment route

**You will build:** the decision procedure Article 43 actually is — which conformity
assessment route a high-risk system must travel, whether the evidence that route demands is
present, in date and traceable, and the EU declaration of conformity and CE marking record
that come out the other end. Plus the thing that quietly invalidates all of it: a
substantial modification.

**Time:** ~80 minutes · **Runs on:** a laptop CPU, no download, no network
· **Prerequisites:** `T10-L01-ai-act-conformity-pack`, `P01-L01-article-12-logging`,
`P01-L02-risk-classification`

P01-L02 left you with a sealed classification record. This lesson reads that record — the
same eight keys, the same question trail — and turns it into a route. Then it does the
thing a compliance pack is for: it refuses to produce a declaration it cannot stand behind.

By the end you will be able to:

1. Implement the Annex III point lookup over P01-L02's record, and explain why a record
   whose point cannot be read is an undetermined route rather than a default one.
2. Implement the four standards gaps of Article 43(1) and the route decision over them,
   including the asymmetry that makes the same gap decisive for one system and irrelevant
   for another.
3. Implement `is_substantial_modification()` from Article 3(23) and Article 43(4), and
   measure how many changes in a log are substantial and how many of those post-date the
   assessment on file.
4. Implement `readiness()` on top of the completeness checker you built in P01-L01, rather
   than a second one, and separate missing evidence from stale evidence from untraceable
   evidence.
5. Implement the Article 47 declaration and the Article 48 CE marking record, each of which
   refuses to exist while the thing before it in the chain does not.

> **This is engineering, not legal advice.** Every article number, quotation and date below
> is sourced in `claims.yaml` with its URL and access date. Three things in this lesson are
> labelled as *this lesson's reading* rather than as sourced fact — sections 4, 6 and 7
> say which and why — and one open question is recorded as unanswered rather than guessed.
> The systems are fictional. For a real system, read the Official Journal text and take
> professional advice.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import hashlib
import io
import json
import re
import sys
import traceback
from datetime import date
from typing import Any, Callable

print("python", sys.version.split()[0], "· standard library only")

# The date this assessment is run. Fixed, so every date this lesson computes is reproducible.
AS_OF = date(2026, 9, 22)
print("as of", AS_OF.isoformat())


def canonical_bytes(obj: Any) -> bytes:
    """One deterministic serialisation of an object. Given to you — you built it in P01-L01."""
    return json.dumps(obj, sort_keys=True, separators=(",", ":")).encode("utf-8")


def seal(record: dict) -> dict:
    """Add P01-L02's `record_hash` over the record without it. Given to you.

    Example:
        >>> len(seal({"a": 1})["record_hash"])
        64
    """
    body = {k: v for k, v in record.items() if k != "record_hash"}
    return dict(body, record_hash=hashlib.sha256(canonical_bytes(body)).hexdigest())


def parse_date(value: Any) -> date | None:
    """An ISO date string, or None for a missing one. Given to you — P01-L02's helper."""
    if not value:
        return None
    return date.fromisoformat(str(value))


def plus_years(day: date, years: int) -> date:
    """`day` shifted by whole years, clamping 29 February onto 28 February. Given to you."""
    try:
        return day.replace(year=day.year + years)
    except ValueError:                       # 29 February in a year that has none
        return day.replace(year=day.year + years, day=28)


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("annex_iii_point",),
    "exercise 2": ("standards_gap",),
    "exercise 3": ("assessment_route",),
    "exercise 4": ("is_substantial_modification",),
    "exercise 5": ("modification_history",),
    "exercise 6": ("readiness",),
    "exercise 7": ("declaration_of_conformity",),
    "exercise 8": ("ce_marking_record",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (<its function>)"; several -> "exercises 3, 6 and 7"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. What Article 43 decides, and what it does not

Article 43 does not ask how risky your system is. P01-L02 settled that. It asks one
narrower question: **who has to look at the evidence before the system goes on the market —
you, or a notified body?**

There are three answers and three non-answers, and which one you get is almost entirely
mechanical:

* **Annex VI, internal control.** You assess your own quality management system and your
  own technical documentation. No third party is involved.
* **Annex VII, with a notified body.** A designated body assesses your quality management
  system and your technical documentation and issues a certificate.
* **The product's own procedure.** An Annex I system folds into the conformity assessment
  its product-safety legislation already requires, with Chapter III Section 2 assessed
  inside it.

The three non-answers: a system that is not high-risk has no Article 43 route at all; a
prohibited practice has no route because it must not be placed on the market; and a record
that does not say which Annex III point applies leaves the route undetermined until
somebody fixes the record. Count them in the cell below rather than here.

In [ ]:
ROUTES = (
    "annex_vi_internal_control",
    "annex_vii_notified_body",
    "annex_i_product_procedure",
    "not_high_risk",
    "none",
    "undetermined",
)
ROUTE_LABEL = {
    "annex_vi_internal_control": "Annex VI — internal control, no notified body",
    "annex_vii_notified_body": "Annex VII — QMS and technical documentation, notified body",
    "annex_i_product_procedure": "the product legislation's own procedure (Article 43(3))",
    "not_high_risk": "no Article 43 route: the system is not high-risk",
    "none": "no route: a prohibited practice may not be placed on the market",
    "undetermined": "the record does not say enough to choose a route",
}
_real = [r for r in ROUTES if ROUTE_LABEL[r].startswith(("Annex", "the product"))]
print(f"{len(ROUTES)} outcomes, of which {len(_real)} are actual assessment procedures "
      f"and {len(ROUTES) - len(_real)} are reasons there is no procedure to run:")
for _r in ROUTES:
    print(f"  {_r:28s} {ROUTE_LABEL[_r]}")

### The asymmetry this lesson is built around

Article 43(1) applies to **point 1 of Annex III** — the biometrics point. There, whether
you applied the harmonised standards decides whether you may choose internal control at
all: apply them in full and you pick; fall short and Annex VII is compulsory.

Article 43(2) applies to **points 2 to 8** — critical infrastructure, education,
employment, essential private and public services, law enforcement, migration, justice.
There it is Annex VI, full stop,
"which does not provide for the involvement of a notified body". No standards test. No
choice. Two systems with an identical standards gap can therefore land on different routes,
and the thing that separates them is a single digit in Annex III.

You will see that happen on screen in section 5. Do not take it from this paragraph.

In [ ]:
# The four cases in which Article 43(1) forces the Annex VII route. Ids, not sentences, so
# the rubric grades the outcome rather than your prose.
GAP_NO_STANDARD = "art43_1_a_no_standard_or_common_specification"
GAP_NOT_FULLY_APPLIED = "art43_1_b_standard_not_applied_or_partial"
GAP_COMMON_SPEC_NOT_APPLIED = "art43_1_c_common_specification_not_applied"
GAP_RESTRICTED = "art43_1_d_standard_published_with_restriction"
STANDARDS_GAPS = (GAP_NO_STANDARD, GAP_NOT_FULLY_APPLIED,
                  GAP_COMMON_SPEC_NOT_APPLIED, GAP_RESTRICTED)

# Reasons an `assessment_route` outcome was reached.
REASON_PROHIBITED = "prohibited_practice"
REASON_NOT_HIGH_RISK = "not_high_risk_tier"
REASON_ANNEX_I = "annex_i_product_legislation"
REASON_POINT_1_CHOICE = "annex_iii_point_1_provider_choice"
REASON_POINT_1_STANDARDS_GAP = "annex_iii_point_1_standards_gap"
REASON_POINTS_2_TO_8 = "annex_iii_points_2_to_8_internal_control"
REASON_POINT_UNKNOWN = "annex_iii_point_not_determined"

# Article 43(1), third subparagraph: for these deployers the market surveillance authority
# acts as the notified body rather than a designated private body.
AUTHORITY_DEPLOYERS = ("law_enforcement", "immigration", "asylum", "union_institution")

print(f"{len(STANDARDS_GAPS)} standards gaps, {len(AUTHORITY_DEPLOYERS)} deployer kinds for "
      "whom the market surveillance authority acts as the notified body")

## 2. The records P01-L02 handed you

These are classification records in exactly the shape P01-L02 sealed them: eight keys, a
question trail of `{question, answer, decisive}` dicts, and a `record_hash` over everything
else. Nothing here re-derives a tier — that argument is settled and this lesson consumes
the result.

Read the `q3_annex_iii_area` answers. The Annex III **point** decides the whole route, and
P01-L02 stored it as free text inside a human-readable area label, because that is what a
classification interview records. One of the nine records does not carry a readable point
at all. That record is not a rounding error; it is the common case in a real pack.

In [ ]:
def _record(system_id: str, tier: str, trail: list, derogation: Any = None,
            obligations: tuple = (), deadlines: dict | None = None) -> dict:
    """Build one P01-L02-shaped classification record and seal it. Given to you."""
    return seal({
        "system_id": system_id,
        "as_of": AS_OF.isoformat(),
        "risk_tier": tier,
        "question_trail": [{"question": q, "answer": a, "decisive": d} for q, a, d in trail],
        "derogation": derogation,
        "obligations": sorted(obligations),
        "deadlines": dict(deadlines or {}),
    })


_HIGH_RISK_III = ("art4_ai_literacy", "ch3_high_risk_annex_iii")
_DEADLINES_III = {"art4_ai_literacy": "2025-02-02", "ch3_high_risk_annex_iii": "2027-12-02"}


def _annex_iii_trail(area: str, q4_reason: str) -> list:
    """The trail P01-L02 records for a system that named an Annex III area."""
    return [("q1_prohibited", False, False),
            ("q2_annex_i_safety_component", False, False),
            ("q3_annex_iii_area", area, False),
            ("q4_article_6_3", q4_reason, True)]


RECORDS = {
    "mood-meter": _record(
        "mood-meter", "prohibited",
        [("q1_prohibited", True, True)],
        obligations=("art4_ai_literacy", "art5_prohibited_practice"),
        deadlines={"art4_ai_literacy": "2025-02-02", "art5_prohibited_practice": "2025-02-02"}),
    "lift-door-sensor": _record(
        "lift-door-sensor", "high_risk_annex_i",
        [("q1_prohibited", False, False), ("q2_annex_i_safety_component", True, True)],
        obligations=("art4_ai_literacy", "ch3_high_risk_annex_i"),
        deadlines={"art4_ai_literacy": "2025-02-02", "ch3_high_risk_annex_i": "2028-08-02"}),
    "face-match": _record(
        "face-match", "high_risk_annex_iii",
        _annex_iii_trail("remote biometric identification (Annex III point 1(a))", "not_claimed"),
        obligations=_HIGH_RISK_III, deadlines=_DEADLINES_III),
    "border-biometrics": _record(
        "border-biometrics", "high_risk_annex_iii",
        _annex_iii_trail("biometric categorisation (Annex III point 1(b))", "not_claimed"),
        obligations=_HIGH_RISK_III, deadlines=_DEADLINES_III),
    "credit-scorer": _record(
        "credit-scorer", "high_risk_annex_iii",
        _annex_iii_trail("creditworthiness (Annex III point 5(b))", "profiling_override"),
        obligations=_HIGH_RISK_III, deadlines=_DEADLINES_III),
    "loan-copilot": _record(
        "loan-copilot", "high_risk_annex_iii",
        _annex_iii_trail("creditworthiness (Annex III point 5(b))", "profiling_override"),
        obligations=_HIGH_RISK_III, deadlines=_DEADLINES_III),
    "cv-ranker": _record(
        "cv-ranker", "high_risk_annex_iii",
        _annex_iii_trail("employment (Annex III point 4(a))", "profiling_override"),
        obligations=_HIGH_RISK_III, deadlines=_DEADLINES_III),
    "shift-note-tidier": _record(
        "shift-note-tidier", "annex_iii_derogated",
        _annex_iii_trail("workers' management (Annex III point 4(b))", "granted"),
        derogation={"claimed": True, "granted": True, "condition": "narrow_procedural_task",
                    "reason": "granted", "documentation_gaps": []},
        obligations=("art4_ai_literacy", "art6_4_derogation_documentation",
                     "art49_2_registration"),
        deadlines={"art4_ai_literacy": "2025-02-02",
                   "art6_4_derogation_documentation": "2027-12-02",
                   "art49_2_registration": "2027-12-02"}),
    "legacy-sorter": _record(
        "legacy-sorter", "high_risk_annex_iii",
        _annex_iii_trail("sorting inbound benefit applications for a caseworker queue",
                         "not_claimed"),
        obligations=_HIGH_RISK_III, deadlines=_DEADLINES_III),
}

_tiers = sorted({r["risk_tier"] for r in RECORDS.values()})
print(f"{len(RECORDS)} records across {len(_tiers)} tiers: {', '.join(_tiers)}")
print(f"every record seals cleanly: "
      f"{all(seal(r)['record_hash'] == r['record_hash'] for r in RECORDS.values())}")
for _sid, _rec in RECORDS.items():
    _q3 = next((s["answer"] for s in _rec["question_trail"]
                if s["question"] == "q3_annex_iii_area"), None)
    print(f"  {_sid:20s} {_rec['risk_tier']:22s} {_q3 or '—'}")

## 3. Exercise 1 — `annex_iii_point(record)`

Everything downstream turns on one integer. Pull it out of the record, refuse to invent it,
and refuse to accept one Annex III does not have.

The regex is given to you; the exercise is the record traversal and the range check. Note
what the range check is really for: a record reading "Annex III point 9" is not a system in
some ninth category, it is a record somebody typed wrong, and a route chosen from it would
be a route chosen from a typo.

<details><summary>💡 Hint 1 — what to think about</summary>

Several records here must come back None, for different reasons: no q3 step at all, a q3
answer that is empty, an area label that names no point, and a number Annex III does not
have. Each is a record that does not determine a route, not a system in some default
category. One trap hides in `str()`: think about what `str(None)` looks like by the time a
pattern sees it.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Walk the trail and stop at the FIRST step whose question is the q3 one; if there is none,
return None. If its answer is falsy, return None before the regex ever runs. Search
`str(answer)` with the given pattern, and treat no match as None. Convert the captured digits
to an `int` and return it only when it is in `ANNEX_III_POINTS`. Never fall back on the tier,
and ignore the step's `decisive` flag.
</details>

In [ ]:
# Given to you: the pattern that finds a point number inside P01-L02's free-text area label.
ANNEX_III_POINT_PATTERN = re.compile(r"annex\s+iii\s+point\s+(\d+)", re.IGNORECASE)
ANNEX_III_POINTS = tuple(range(1, 9))     # Annex III lists eight points

print(f"Annex III has {len(ANNEX_III_POINTS)} points: "
      f"{ANNEX_III_POINTS[0]} to {ANNEX_III_POINTS[-1]}")
print("pattern demo:",
      ANNEX_III_POINT_PATTERN.search("creditworthiness (Annex III point 5(b))").group(1))

In [ ]:
def annex_iii_point(record: dict) -> int | None:
    """Read the Annex III point number out of a P01-L02 classification record.

    Walk `record["question_trail"]` and take the FIRST step whose "question" is
    "q3_annex_iii_area". Its "answer" is the free-text area label P01-L02 stored, or None.

    Return None when: there is no q3 step at all; the q3 answer is falsy; the pattern does
    not match; or the matched number is not one of ANNEX_III_POINTS. Otherwise return the
    number as an int.

    Use ANNEX_III_POINT_PATTERN.search() on str(answer). Ignore the step's "decisive" flag
    entirely — P01-L02 records q3 as never decisive, and that has nothing to do with whether
    the area it names is readable.

    Example:
        >>> annex_iii_point(RECORDS["credit-scorer"]), annex_iii_point(RECORDS["mood-meter"])
        (5, None)

    Returns:
        int in ANNEX_III_POINTS, or None when the record does not determine one.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_annex_iii_point() -> None:
    got = {sid: annex_iii_point(rec) for sid, rec in RECORDS.items()}
    assert got["credit-scorer"] == 5 and got["loan-copilot"] == 5, (
        f"credit-scorer came back {got['credit-scorer']!r} — the q3 answer reads "
        "'creditworthiness (Annex III point 5(b))', so the point is the int 5, not the "
        "string '5' and not the sub-point letter.")
    assert got["face-match"] == 1 and got["border-biometrics"] == 1, (
        f"both biometrics systems are point 1; got {got['face-match']!r} and "
        f"{got['border-biometrics']!r}.")
    assert got["cv-ranker"] == 4 and got["shift-note-tidier"] == 4, (
        "employment and workers' management are both Annex III point 4.")
    assert got["mood-meter"] is None and got["lift-door-sensor"] is None, (
        "neither record has a q3 step at all — a missing question is None, not an error and "
        "not a default point.")
    assert got["legacy-sorter"] is None, (
        f"legacy-sorter's area label names no point, so it is None; got {got['legacy-sorter']!r}. "
        "Do not fall back to a point because the tier says high-risk.")

    out_of_range = _record("x", "high_risk_annex_iii",
                           _annex_iii_trail("a made-up area (Annex III point 9)", "not_claimed"))
    assert annex_iii_point(out_of_range) is None, (
        "Annex III point 9 does not exist; a number outside ANNEX_III_POINTS is None, not 9.")
    blank = _record("x", "high_risk_annex_iii", _annex_iii_trail(None, "not_claimed"))
    assert annex_iii_point(blank) is None, (
        "a q3 step whose answer is None must give None — check you are not calling "
        ".search() on the string 'None'.")
    print("exercise 1 looks right")


_try("exercise 1", _check_annex_iii_point)

## 4. Exercise 2 — `standards_gap(context)`

Article 43(1)'s second subparagraph lists four situations in which the Annex VII route is
compulsory for a point 1 system. They are not alternatives to one another: two of them can
bite at once, and an implementation that returns the first one it finds loses the second
reason a reviewer would have wanted to read.

**This lesson's first flagged reading.** Point (c) says the common specifications "exist, but
the provider has not applied them", and point (a) refers to them being "not available".
This lesson reads (c) as exactly what it says — common specifications exist and were not
applied — independently of whether a harmonised standard was also applied. That is a
reading of a cross-reference, not a ruling.

<details><summary>💡 Hint 1 — what to think about</summary>

The four gaps are not a ladder, so a chain that returns at the first match silently loses a
reason a reviewer would want to read. Notice too which gaps presuppose that something exists:
three of them can only fire when there is a standard or a common specification to misapply,
and one only when there is neither.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Read every field with `.get()`, so an absent key counts as falsy. Test the four conditions
independently and collect the id of each one that holds: (a) needs BOTH the standard and the
specification absent; (b) needs the standard to exist and `applied` to be anything but
`"full"`, a missing value included; (c) needs the specification to exist and not be applied;
(d) needs the standard to exist and carry a restriction. Sort before you return.
</details>

In [ ]:
def standards_gap(context: dict) -> list:
    """Return the sorted ids of the Article 43(1) standards gaps this context exhibits.

    Test all four independently and collect every one that fires:

      GAP_NO_STANDARD             neither context["harmonised_standard_exists"] nor
                                  context["common_specification_exists"] is truthy.
      GAP_NOT_FULLY_APPLIED       context["harmonised_standard_exists"] is truthy AND
                                  context["harmonised_standard_applied"] != "full".
      GAP_COMMON_SPEC_NOT_APPLIED context["common_specification_exists"] is truthy AND
                                  context["common_specification_applied"] is falsy.
      GAP_RESTRICTED              context["harmonised_standard_exists"] is truthy AND
                                  context["harmonised_standard_restricted"] is truthy.

    Read every field with .get() so a context that omits one is treated as falsy rather than
    raising. "full" is the only value of harmonised_standard_applied that closes gap (b);
    "partial" and "none" both leave it open.

    Example:
        >>> standards_gap({"harmonised_standard_exists": True,
        ...                "harmonised_standard_applied": "full"})
        []

    Returns:
        list[str], sorted, drawn from STANDARDS_GAPS; empty when nothing is wrong.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_standards_gap() -> None:
    clean = {"harmonised_standard_exists": True, "harmonised_standard_applied": "full",
             "harmonised_standard_restricted": False, "common_specification_exists": False,
             "common_specification_applied": False}
    assert standards_gap(clean) == [], (
        f"a standard that exists, was applied in full and carries no restriction is no gap; "
        f"got {standards_gap(clean)}.")
    assert standards_gap({}) == [GAP_NO_STANDARD], (
        "an empty context has no standard and no common specification, which is gap (a) and "
        "only gap (a): (b), (c) and (d) all require something to exist first.")
    partial = dict(clean, harmonised_standard_applied="partial")
    assert standards_gap(partial) == [GAP_NOT_FULLY_APPLIED], (
        "'applied only part of the harmonised standard' is gap (b); only 'full' closes it.")
    restricted = dict(clean, harmonised_standard_restricted=True)
    assert standards_gap(restricted) == [GAP_RESTRICTED], (
        "a standard published with a restriction is gap (d) even when it was applied in full.")
    both = dict(clean, harmonised_standard_applied="none", harmonised_standard_restricted=True)
    assert standards_gap(both) == sorted([GAP_NOT_FULLY_APPLIED, GAP_RESTRICTED]), (
        f"got {standards_gap(both)} — (b) and (d) are independent tests, so both fire here. "
        "Returning the first gap you find throws away the second reason.")
    spec_only = {"harmonised_standard_exists": False, "common_specification_exists": True,
                 "common_specification_applied": False}
    assert standards_gap(spec_only) == [GAP_COMMON_SPEC_NOT_APPLIED], (
        "a common specification exists here, so gap (a) does NOT fire — (a) needs both to be "
        "absent — and (c) does.")
    print("exercise 2 looks right")


_try("exercise 2", _check_standards_gap)

## 5. Exercise 3 — `assessment_route(record, context)`

The context is what the classification record does not know: which standards the provider
actually applied, which route it would choose if it had a choice, and who is going to put
the system into service. Two of the nine contexts below carry the *same* standards gap. One
of them is forced onto Annex VII by it and the other is not affected by it at all.

<details><summary>💡 Hint 1 — what to think about</summary>

Article 43(1) is the longer, more interesting paragraph, and it applies to the fewest
systems. Before you run any standards test, ask which Annex III point you are on: for points
2 to 8 the gap is never consulted, and reporting one there suggests it mattered. For point 1,
ask WHEN the provider's stated preference counts, and what a preference naming something
other than an Annex III procedure should become.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Branch on the tier first, in the documented order. For an Annex III record, read the point
with your exercise 1 function and stop at undetermined when it is None — never a default
route. For point 1, compute the gaps once: with none, honour `chosen_route` only if it is one
of the two Annex III procedures, else internal control; with any, the notified-body route is
forced. Points 2 to 8 get internal control and an empty gap list. Derive
`notified_body_required` from the route (plus the Annex I context flag), then the role from
the deployer against `AUTHORITY_DEPLOYERS`.
</details>

In [ ]:
def _context(standard: bool = True, applied: str = "full", restricted: bool = False,
             spec: bool = False, spec_applied: bool = False,
             chosen: str | None = None, deployer: str | None = None) -> dict:
    """Build one provider context. Given to you."""
    return {"harmonised_standard_exists": standard, "harmonised_standard_applied": applied,
            "harmonised_standard_restricted": restricted,
            "common_specification_exists": spec, "common_specification_applied": spec_applied,
            "chosen_route": chosen, "deployer_authority": deployer}


CONTEXTS = {
    "mood-meter": _context(standard=False, applied="none"),
    "lift-door-sensor": _context(),
    # Point 1, standards applied in full: the provider genuinely chooses, and chooses Annex VI.
    "face-match": _context(chosen="annex_vi_internal_control"),
    # Point 1, standard applied only in part, and an immigration deployer. The provider asks
    # for Annex VI and does not get it.
    "border-biometrics": _context(applied="partial", chosen="annex_vi_internal_control",
                                  deployer="immigration"),
    # Point 5. The SAME gap as border-biometrics, and it changes nothing.
    "credit-scorer": _context(applied="partial", chosen="annex_vii_notified_body"),
    "loan-copilot": _context(standard=False, applied="none"),
    "cv-ranker": _context(),
    "shift-note-tidier": _context(),
    "legacy-sorter": _context(standard=False, applied="none"),
}

print(f"{len(CONTEXTS)} contexts; "
      f"{sum(1 for c in CONTEXTS.values() if c['chosen_route'])} express a route preference, "
      f"{sum(1 for c in CONTEXTS.values() if c['deployer_authority'])} name an authority "
      "deployer")

In [ ]:
def assessment_route(record: dict, context: dict) -> dict:
    """Decide the Article 43 conformity assessment route for one classified system.

    Work in this order and stop at the first rule that applies.

      1. record["risk_tier"] == "prohibited"
         -> route "none", reason REASON_PROHIBITED, basis "Article 5".
      2. record["risk_tier"] == "high_risk_annex_i"
         -> route "annex_i_product_procedure", reason REASON_ANNEX_I, basis "Article 43(3)".
            notified_body_required is bool(context.get("annex_i_notified_body_involved")):
            whether the product legislation already puts a body in the loop is a fact about
            that legislation, not something Article 43 decides.
      3. record["risk_tier"] == "high_risk_annex_iii":
         point = annex_iii_point(record)
         3a. point is None -> route "undetermined", reason REASON_POINT_UNKNOWN,
             basis "Article 43(1)-(2)", notified_body_required False.
         3b. point == 1 and standards_gap(context) is EMPTY -> the provider chooses:
             route = context["chosen_route"] when that is one of the two Annex III procedures
             ("annex_vi_internal_control" or "annex_vii_notified_body"), and
             "annex_vi_internal_control" when it is missing or is anything else.
             Reason REASON_POINT_1_CHOICE, basis "Article 43(1)".
         3c. point == 1 and standards_gap(context) is NON-EMPTY
             -> route "annex_vii_notified_body", reason REASON_POINT_1_STANDARDS_GAP,
             basis "Article 43(1) second subparagraph".
         3d. point in 2..8 -> route "annex_vi_internal_control", reason REASON_POINTS_2_TO_8,
             basis "Article 43(2)". The standards gap is NOT consulted: Article 43(2) is
             unconditional, and a point 5 system with no harmonised standard at all still
             travels internal control.
      4. any other tier -> route "not_high_risk", reason REASON_NOT_HIGH_RISK,
         basis "Article 43 (applies to high-risk AI systems)".

    "gaps" is standards_gap(context) for an Annex III point 1 system and [] for every other
    outcome, including point 2..8: reporting a gap that did not bear on the decision invites
    the next reader to think it did.

    "notified_body_required" is True exactly when the route is "annex_vii_notified_body",
    plus the Annex I case in rule 2. "notified_body_role" is None when no body is required,
    "market_surveillance_authority" when context["deployer_authority"] is in
    AUTHORITY_DEPLOYERS, and "notified_body" otherwise.

    Example:
        >>> r = assessment_route(RECORDS["credit-scorer"], CONTEXTS["credit-scorer"])
        >>> r["route"], r["notified_body_required"], r["gaps"]
        ('annex_vi_internal_control', False, [])

    Returns:
        dict with exactly these nine keys:
          "system_id"              record["system_id"]
          "risk_tier"              record["risk_tier"]
          "annex_iii_point"        int | None
          "route"                  str, one of ROUTES
          "gaps"                   list[str], sorted
          "notified_body_required" bool
          "notified_body_role"     str | None
          "reason"                 str, one of the REASON_* ids
          "basis"                  str, the article the decision rests on
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_assessment_route() -> None:
    got = assessment_route(RECORDS["face-match"], CONTEXTS["face-match"])
    assert set(got) == {"system_id", "risk_tier", "annex_iii_point", "route", "gaps",
                        "notified_body_required", "notified_body_role", "reason", "basis"}, (
        f"keys were {sorted(got)} — return exactly the nine documented names.")
    assert got["route"] == "annex_vi_internal_control" and got["gaps"] == [], (
        f"face-match applied the standard in full, so it chooses, and it chose Annex VI; "
        f"got {got['route']!r} with gaps {got['gaps']}.")
    assert got["reason"] == REASON_POINT_1_CHOICE and got["notified_body_required"] is False, (
        "a point 1 system with no gap reaches Annex VI by CHOICE, and internal control "
        "involves no notified body.")

    border = assessment_route(RECORDS["border-biometrics"], CONTEXTS["border-biometrics"])
    assert border["route"] == "annex_vii_notified_body", (
        f"border-biometrics asked for Annex VI but applied only part of the standard, so "
        f"Article 43(1) makes Annex VII compulsory; got {border['route']!r}. The provider's "
        "stated preference is only consulted when the gap list is empty.")
    assert border["notified_body_role"] == "market_surveillance_authority", (
        f"role came back {border['notified_body_role']!r} — an immigration deployer puts the "
        "market surveillance authority in the notified body's chair.")

    credit = assessment_route(RECORDS["credit-scorer"], CONTEXTS["credit-scorer"])
    assert credit["route"] == "annex_vi_internal_control", (
        f"credit-scorer has the SAME standards gap as border-biometrics and asked for Annex "
        f"VII, and still travels Annex VI: Article 43(2) is unconditional for points 2 to 8. "
        f"Got {credit['route']!r} — you are applying the Article 43(1) test outside point 1.")
    assert credit["gaps"] == [] and credit["reason"] == REASON_POINTS_2_TO_8, (
        f"gaps came back {credit['gaps']} — the gap did not bear on a points 2-to-8 decision, "
        "so reporting it invites the next reader to think it did.")

    for sid, expected in (("mood-meter", "none"), ("shift-note-tidier", "not_high_risk"),
                          ("legacy-sorter", "undetermined"),
                          ("lift-door-sensor", "annex_i_product_procedure")):
        route = assessment_route(RECORDS[sid], CONTEXTS[sid])["route"]
        assert route == expected, f"{sid} should route to {expected!r}, got {route!r}."

    embedded = assessment_route(RECORDS["lift-door-sensor"],
                                dict(CONTEXTS["lift-door-sensor"],
                                     annex_i_notified_body_involved=True))
    assert embedded["notified_body_required"] is True, (
        "an Annex I system whose product legislation already involves a notified body must "
        "report that, even though Article 43(3) did not choose the body.")
    print("exercise 3 looks right")


_try("exercise 3", _check_assessment_route)

In [ ]:
def _show_routes() -> None:
    """The route table. Read the two rows that share a gap and do not share a route."""
    print(f"{'system':20s} {'pt':>3s} {'route':28s} {'NB':>3s} reason")
    for sid, rec in RECORDS.items():
        r = assessment_route(rec, CONTEXTS[sid])
        point = "—" if r["annex_iii_point"] is None else str(r["annex_iii_point"])
        print(f"  {sid:18s} {point:>3s} {r['route']:28s} "
              f"{'yes' if r['notified_body_required'] else ' no':>3s} {r['reason']}")
    twins = ["border-biometrics", "credit-scorer"]
    gaps = {s: standards_gap(CONTEXTS[s]) for s in twins}
    routes = {s: assessment_route(RECORDS[s], CONTEXTS[s])["route"] for s in twins}
    print(f"\n{twins[0]} and {twins[1]} share the gap {gaps[twins[0]]} "
          f"(identical: {gaps[twins[0]] == gaps[twins[1]]}) and land on "
          f"{len(set(routes.values()))} different routes: "
          f"{routes[twins[0]]} vs {routes[twins[1]]}")


_try("route table", _show_routes, needs=("exercise 1", "exercise 2", "exercise 3"))

## 6. Exercise 4 — `is_substantial_modification(change, plan)`

This is the rule that quietly voids a pack. Article 3(23) defines a substantial
modification as a change **after** placing on the market that was **not foreseen or
planned** in the initial conformity assessment, and that either affects compliance with
Chapter III Section 2 or modifies the intended purpose. Article 43(4) then carves out the
changes a continuously-learning system was always going to make — but only those
pre-determined by the provider **and** carried in the Annex IV point 2(f) description.

Pre-determined is not enough on its own. A plan in somebody's head is not in the technical
documentation, and the carve-out is drafted around what the documentation contains.

**This lesson's second flagged reading.** Article 43(4)'s carve-out is written about
"changes to the high-risk AI system and its performance". A change to the *intended
purpose* is a different animal, and Article 3(23) names it as its own limb. So this lesson
tests intended purpose **before** the carve-out: you cannot pre-declare your way into a new
purpose. That is a reading of where a sentence stops, not a ruling.

<details><summary>💡 Hint 1 — what to think about</summary>

The order of the five rules IS the exercise. What should a change the provider pre-determined
but never wrote into the Annex IV point 2(f) description fall through to? Can a change of
intended purpose shelter under the carve-out at all? And the boundary: is a change dated on
the very day of placing "before" it?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Parse both dates with `parse_date()`. Rule 1 fires only when a placing date exists and the
change is strictly earlier. Then the purpose flag, then the carve-out — the id in
`pre_determined` AND the 2(f) flag set, both of them — then the compliance flag, then the
no-effect default. Return at the first rule that fires, and pick the basis by which rule it
was.
</details>

In [ ]:
REASON_BEFORE_PLACING = "before_placing_on_the_market"
REASON_INTENDED_PURPOSE = "intended_purpose_modified"
REASON_PRE_DETERMINED = "pre_determined_and_documented"
REASON_COMPLIANCE_AFFECTED = "chapter_iii_compliance_affected"
REASON_NO_EFFECT = "no_effect_on_compliance_or_purpose"

MODIFICATION_REASONS = (REASON_BEFORE_PLACING, REASON_INTENDED_PURPOSE, REASON_PRE_DETERMINED,
                        REASON_COMPLIANCE_AFFECTED, REASON_NO_EFFECT)
_SUBSTANTIAL_REASONS = (REASON_INTENDED_PURPOSE, REASON_COMPLIANCE_AFFECTED)
print(f"{len(MODIFICATION_REASONS)} verdicts a change can attract, of which "
      f"{len(_SUBSTANTIAL_REASONS)} are substantial")

In [ ]:
def is_substantial_modification(change: dict, plan: dict) -> dict:
    """Decide whether one logged change is a substantial modification.

    Work in this order and stop at the first rule that applies. The order is the exercise.

      1. change["on"] is STRICTLY EARLIER than plan["placed_on_market"]
         -> not substantial, REASON_BEFORE_PLACING. Article 3(23) is about a change "after
            its placing on the market or putting into service"; before that, the change is
            part of what the initial assessment assessed.
      2. change["modifies_intended_purpose"] truthy
         -> substantial, REASON_INTENDED_PURPOSE. Before the carve-out, deliberately: see
            the note above this cell.
      3. change["change_id"] is in plan["pre_determined"] AND
         plan["documented_in_annex_iv_2f"] is truthy
         -> not substantial, REASON_PRE_DETERMINED. BOTH halves are required. A change the
            provider pre-determined but never wrote into the Annex IV point 2(f) description
            falls through to rule 4.
      4. change["affects_chapter_iii_compliance"] truthy
         -> substantial, REASON_COMPLIANCE_AFFECTED.
      5. otherwise -> not substantial, REASON_NO_EFFECT.

    Compare the dates with parse_date(), not as strings, and treat a missing
    plan["placed_on_market"] as "no placing date on file", which means rule 1 cannot fire.

    Example:
        >>> plan = {"placed_on_market": "2026-06-01", "pre_determined": (),
        ...         "documented_in_annex_iv_2f": False}
        >>> is_substantial_modification(
        ...     {"change_id": "C1", "on": "2026-07-01", "modifies_intended_purpose": False,
        ...      "affects_chapter_iii_compliance": True}, plan)["substantial"]
        True

    Returns:
        dict with exactly these five keys:
          "change_id"   change["change_id"]
          "on"          change["on"]
          "substantial" bool
          "reason"      str, one of the MODIFICATION_REASONS
          "basis"       "Article 43(4)" for rules 1 and 3, "Article 3(23)" otherwise
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_is_substantial() -> None:
    plan = {"placed_on_market": "2026-06-01", "assessed_on": "2026-07-30",
            "pre_determined": ("C-PRE",), "documented_in_annex_iv_2f": True}

    def change(cid, on, compliance=False, purpose=False):
        return {"change_id": cid, "on": on, "description": "t",
                "affects_chapter_iii_compliance": compliance,
                "modifies_intended_purpose": purpose}

    got = is_substantial_modification(change("C1", "2026-05-01", compliance=True), plan)
    assert set(got) == {"change_id", "on", "substantial", "reason", "basis"}, (
        f"keys were {sorted(got)} — return exactly the five documented names.")
    assert got["substantial"] is False and got["reason"] == REASON_BEFORE_PLACING, (
        "a change before the system was placed on the market is part of what the initial "
        f"assessment assessed, whatever it touched; got {got['reason']!r}.")

    on_the_day = is_substantial_modification(change("C2", "2026-06-01", compliance=True), plan)
    assert on_the_day["substantial"] is True, (
        "rule 1 is STRICTLY earlier: a change dated the same day as the placing is not "
        "before it.")

    undocumented = dict(plan, documented_in_annex_iv_2f=False)
    got = is_substantial_modification(change("C-PRE", "2026-08-01", compliance=True),
                                      undocumented)
    assert got["substantial"] is True and got["reason"] == REASON_COMPLIANCE_AFFECTED, (
        "pre-determined is only half of Article 43(4): without the Annex IV point 2(f) "
        f"description the carve-out does not apply. Got {got['reason']!r}.")
    got = is_substantial_modification(change("C-PRE", "2026-08-01", compliance=True), plan)
    assert got["substantial"] is False and got["reason"] == REASON_PRE_DETERMINED, (
        "pre-determined AND documented is the carve-out; this one must not be substantial.")

    got = is_substantial_modification(change("C-PRE", "2026-08-01", purpose=True), plan)
    assert got["substantial"] is True and got["reason"] == REASON_INTENDED_PURPOSE, (
        "the intended-purpose test runs BEFORE the carve-out — you cannot pre-declare your "
        f"way into a new purpose. Got {got['reason']!r}.")

    got = is_substantial_modification(change("C9", "2026-08-01"), plan)
    assert got["substantial"] is False and got["reason"] == REASON_NO_EFFECT, (
        "a change that touches neither compliance nor purpose is a change, not a substantial "
        "modification.")
    assert got["basis"] == "Article 3(23)", (
        f"basis came back {got['basis']!r} — rules 1 and 3 rest on Article 43(4); the rest on "
        "the Article 3(23) definition.")
    print("exercise 4 looks right")


_try("exercise 4", _check_is_substantial)

## 7. Exercise 5 — `modification_history(changes, plan)`

One substantial modification does not necessarily reset anything. What resets the route is
a substantial modification the **assessment on file has not seen**. A provider who
re-assessed after the last one is clean; a provider who did not is running on a certificate
for a system that no longer exists.

**This lesson's third flagged reading.** A change dated exactly on `assessed_on` counts as
NOT covered by it. Dates in a change log have day granularity, and a day cannot order two
events inside it, so the lesson takes the reading that leaves the provider re-assessing
rather than the one that lets a same-day change slip under the certificate. That is a
modelling choice about ambiguity, not a rule anybody wrote down.

<details><summary>💡 Hint 1 — what to think about</summary>

Substantial is not the same as uncovered. A provider who re-assessed after a substantial
change is covered for it; only the substantial changes the assessment on file has not seen
reset the route. Decide what a change dated on the assessment day itself counts as (the note
above says), what a plan with no assessment at all means, and how to read the log in time
order without rearranging the caller's list.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Sort a copy of the log on `(on, change_id)` and assess each change with your exercise 4
function. From the substantial verdicts take the sorted ids and the LATEST date, or None when
there are none. A substantial change is uncovered when the plan has no `assessed_on`, or when
its date is on or after `assessed_on`, compared as dates. `route_reset` is simply whether
that uncovered list is non-empty.
</details>

In [ ]:
def modification_history(changes: list, plan: dict) -> dict:
    """Run is_substantial_modification over a change log and decide whether the route resets.

    Assess every change in `changes`, sorted by (change["on"], change["change_id"]) so two
    runs over the same log agree and a log that arrives out of order still reads in time
    order. Do not mutate `changes`.

    "route_reset" is True when at least one SUBSTANTIAL change is dated on or after
    plan["assessed_on"] — see the note above this cell for why "on or after" and not "after".
    A plan with no assessed_on means nothing has been assessed, so every substantial change
    is uncovered and route_reset is True whenever there is one.

    Example:
        >>> plan = {"placed_on_market": "2026-01-01", "assessed_on": "2026-02-01",
        ...         "pre_determined": (), "documented_in_annex_iv_2f": False}
        >>> modification_history([], plan)["route_reset"]
        False

    Returns:
        dict with exactly these five keys:
          "assessed"            list[dict], one is_substantial_modification verdict per
                                change, in the sorted order described above
          "substantial_ids"     list[str], sorted, the ids whose verdict is substantial
          "last_substantial_on" str | None, the LATEST "on" among the substantial ones
          "since_assessment"    list[str], sorted, the substantial ids dated on or after
                                plan["assessed_on"]
          "route_reset"         bool, True exactly when "since_assessment" is non-empty
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_modification_history() -> None:
    plan = {"placed_on_market": "2026-06-01", "assessed_on": "2026-07-30",
            "pre_determined": ("C-PRE",), "documented_in_annex_iv_2f": True}
    log = [
        {"change_id": "C4", "on": "2026-08-14", "description": "new purpose",
         "affects_chapter_iii_compliance": False, "modifies_intended_purpose": True},
        {"change_id": "C1", "on": "2026-05-10", "description": "pre-market",
         "affects_chapter_iii_compliance": True, "modifies_intended_purpose": False},
        {"change_id": "C3", "on": "2026-07-04", "description": "covered by re-assessment",
         "affects_chapter_iii_compliance": True, "modifies_intended_purpose": False},
        {"change_id": "C-PRE", "on": "2026-06-18", "description": "planned drift",
         "affects_chapter_iii_compliance": True, "modifies_intended_purpose": False},
    ]
    before = json.loads(canonical_bytes(log).decode())
    got = modification_history(log, plan)
    assert set(got) == {"assessed", "substantial_ids", "last_substantial_on",
                        "since_assessment", "route_reset"}, (
        f"keys were {sorted(got)} — return exactly the five documented names.")
    assert json.loads(canonical_bytes(log).decode()) == before, (
        "modification_history sorted the caller's list in place — sort a copy.")
    assert [v["change_id"] for v in got["assessed"]] == ["C1", "C-PRE", "C3", "C4"], (
        f"order came back {[v['change_id'] for v in got['assessed']]} — sort by date, then "
        "by change_id, so the log reads in time order however it arrived.")
    assert got["substantial_ids"] == ["C3", "C4"], (
        f"got {got['substantial_ids']} — C1 is pre-market, C-PRE is pre-determined and "
        "documented, and the other two are substantial.")
    assert got["last_substantial_on"] == "2026-08-14", (
        f"got {got['last_substantial_on']!r} — the LATEST substantial change, not the first "
        "one you met and not the latest change of any kind.")
    assert got["since_assessment"] == ["C4"] and got["route_reset"] is True, (
        f"got {got['since_assessment']} — C3 predates the 2026-07-30 assessment, so it is "
        "covered by it; only C4 is uncovered. A history that resets on any substantial "
        "change cannot tell a re-assessed provider from a negligent one.")

    boundary = dict(plan, assessed_on="2026-08-14")
    assert modification_history(log, boundary)["route_reset"] is True, (
        "a substantial change dated exactly on assessed_on counts as uncovered — see the "
        "note above this cell.")
    later = dict(plan, assessed_on="2026-08-15")
    assert modification_history(log, later)["route_reset"] is False, (
        "with the assessment a day later, every substantial change is covered and the route "
        "holds.")
    empty = modification_history([], plan)
    assert empty["route_reset"] is False and empty["last_substantial_on"] is None, (
        "an empty change log resets nothing and has no latest substantial date.")
    print("exercise 5 looks right")


_try("exercise 5", _check_modification_history)

In [ ]:
CHANGE_LOGS = {
    "face-match": [
        {"change_id": "CHG-02", "on": "2026-04-10", "affects_chapter_iii_compliance": True,
         "modifies_intended_purpose": False, "description": "gallery refresh, planned"},
        {"change_id": "CHG-05", "on": "2026-06-22", "affects_chapter_iii_compliance": True,
         "modifies_intended_purpose": False, "description": "threshold recalibration, planned"},
        {"change_id": "CHG-09", "on": "2026-08-01", "affects_chapter_iii_compliance": False,
         "modifies_intended_purpose": False, "description": "operator UI copy"},
    ],
    "border-biometrics": [
        {"change_id": "CHG-21", "on": "2026-05-05", "affects_chapter_iii_compliance": False,
         "modifies_intended_purpose": False, "description": "log retention raised to 12 months"},
    ],
    "credit-scorer": [
        {"change_id": "CHG-33", "on": "2026-07-11", "affects_chapter_iii_compliance": False,
         "modifies_intended_purpose": False, "description": "new report template"},
    ],
    # Pre-determined, but the Annex IV point 2(f) description was never written. Substantial —
    # and already covered, because the provider re-assessed afterwards.
    "loan-copilot": [
        {"change_id": "CHG-77", "on": "2026-05-02", "affects_chapter_iii_compliance": True,
         "modifies_intended_purpose": False, "description": "monthly retrain on fresh files"},
    ],
    # The trap. Everything else about cv-ranker is immaculate.
    "cv-ranker": [
        {"change_id": "CHG-12", "on": "2026-05-10", "affects_chapter_iii_compliance": True,
         "modifies_intended_purpose": False, "description": "pre-market feature cut"},
        {"change_id": "CHG-41", "on": "2026-06-18", "affects_chapter_iii_compliance": True,
         "modifies_intended_purpose": False, "description": "planned quarterly retrain"},
        {"change_id": "CHG-58", "on": "2026-07-04", "affects_chapter_iii_compliance": True,
         "modifies_intended_purpose": False, "description": "new shortlist ranker"},
        {"change_id": "CHG-63", "on": "2026-08-14", "affects_chapter_iii_compliance": False,
         "modifies_intended_purpose": True,
         "description": "also scores internal promotions now"},
    ],
    "lift-door-sensor": [
        {"change_id": "CHG-88", "on": "2026-07-09", "affects_chapter_iii_compliance": True,
         "modifies_intended_purpose": False, "description": "planned sensor firmware bump"},
    ],
    "mood-meter": [],
    "shift-note-tidier": [],
    "legacy-sorter": [],
}

PLANS = {
    "face-match": {"placed_on_market": "2026-03-02", "assessed_on": "2026-02-20",
                   "pre_determined": ("CHG-02", "CHG-05"), "documented_in_annex_iv_2f": True},
    "border-biometrics": {"placed_on_market": "2026-02-16", "assessed_on": "2026-02-02",
                          "pre_determined": (), "documented_in_annex_iv_2f": False},
    "credit-scorer": {"placed_on_market": "2026-04-01", "assessed_on": "2026-03-20",
                      "pre_determined": (), "documented_in_annex_iv_2f": False},
    "loan-copilot": {"placed_on_market": "2026-02-09", "assessed_on": "2026-06-15",
                     "pre_determined": ("CHG-77",), "documented_in_annex_iv_2f": False},
    "cv-ranker": {"placed_on_market": "2026-06-01", "assessed_on": "2026-07-30",
                  "pre_determined": ("CHG-41",), "documented_in_annex_iv_2f": True},
    "lift-door-sensor": {"placed_on_market": "2026-05-12", "assessed_on": "2026-04-28",
                         "pre_determined": ("CHG-88",), "documented_in_annex_iv_2f": True},
    "mood-meter": {"placed_on_market": "2026-03-01", "assessed_on": None,
                   "pre_determined": (), "documented_in_annex_iv_2f": False},
    "shift-note-tidier": {"placed_on_market": "2026-03-15", "assessed_on": None,
                          "pre_determined": (), "documented_in_annex_iv_2f": False},
    "legacy-sorter": {"placed_on_market": "2026-01-05", "assessed_on": None,
                      "pre_determined": (), "documented_in_annex_iv_2f": False},
}


def _show_history() -> None:
    """Substantial is not the same as uncovered. Two systems here prove it in opposite ways."""
    print(f"{'system':20s} {'changes':>7s} {'substantial':>11s} {'uncovered':>9s}  reset")
    for sid in RECORDS:
        h = modification_history(CHANGE_LOGS[sid], PLANS[sid])
        print(f"  {sid:18s} {len(CHANGE_LOGS[sid]):>7d} {len(h['substantial_ids']):>11d} "
              f"{len(h['since_assessment']):>9d}  {h['route_reset']}")
    loan = modification_history(CHANGE_LOGS["loan-copilot"], PLANS["loan-copilot"])
    cv = modification_history(CHANGE_LOGS["cv-ranker"], PLANS["cv-ranker"])
    print(f"\nloan-copilot: {len(loan['substantial_ids'])} substantial "
          f"({', '.join(loan['substantial_ids'])}) but "
          f"{len(loan['since_assessment'])} since the {PLANS['loan-copilot']['assessed_on']} "
          f"assessment -> route holds")
    print(f"cv-ranker:    {len(cv['substantial_ids'])} substantial "
          f"({', '.join(cv['substantial_ids'])}) and "
          f"{len(cv['since_assessment'])} since the {PLANS['cv-ranker']['assessed_on']} "
          f"assessment ({', '.join(cv['since_assessment'])}) -> route resets")


_try("modification table", _show_history, needs=("exercise 4", "exercise 5"))

## 8. The pack, and the completeness checker you already have

A route tells you which evidence has to exist. Whether it does exist is a question you
answered in P01-L01, and `completeness_report()` below is that function, unchanged. It asks
one thing: does some entry in a trace carry this required field with a value that is not
`None`, not blank and not an empty container?

Reusing it is not tidiness. A programme with two completeness checkers has two definitions
of "present", and the day they disagree is the day a pack passes one gate and fails the
other with no way to say which was right. So the pack gets reshaped into a trace and handed
to the checker you already trust.

In [ ]:
def completeness_report(trace: list, required: tuple) -> dict:
    """P01-L01's completeness checker, unchanged. Given to you — do not rewrite it.

    A field is PRESENT when at least one entry carries it, in `entry["event"]` or in
    `entry["event"]["payload"]`, with a value that is not None, not a blank string and not
    an empty container. 0 and False are real values.
    """
    def has_value(value: Any) -> bool:
        if value is None:
            return False
        if isinstance(value, str):
            return bool(value.strip())
        if isinstance(value, (list, tuple, dict, set)):
            return bool(value)
        return True

    present: list = []
    for field in required:
        for entry in trace:
            event = entry["event"]
            if field in event and has_value(event[field]):
                present.append(field)
                break
            payload = event.get("payload", {})
            if field in payload and has_value(payload[field]):
                present.append(field)
                break
    missing = sorted(set(required) - set(present))
    seqs = [entry["seq"] for entry in trace]
    return {"decision_id": None, "present": sorted(present), "missing": missing,
            "coverage": len(present) / len(required) if required else 1.0,
            "complete": not missing,
            "first_seq": min(seqs) if seqs else None,
            "last_seq": max(seqs) if seqs else None}


def pack_as_trace(pack: dict) -> list:
    """Reshape an evidence pack into the trace completeness_report reads. Given to you.

    One entry per evidence item, carrying the item's document reference as its payload
    field — or None when the item is not marked "present", because a planned item is a
    promise and P01-L01 already knows what to do with a None.
    """
    return [{"seq": i,
             "event": {"system_id": pack["system_id"],
                       "payload": {item: (rec.get("document")
                                          if rec.get("status") == "present" else None)}}}
            for i, (item, rec) in enumerate(sorted(pack["evidence"].items()))]

In [ ]:
# The evidence each route demands. The three items MODULES.md assigns to this module —
# the conformity assessment record, the EU declaration of conformity and the CE marking
# record — are OUTPUTS of this lesson, so none of them is an input requirement here.
CORE_EVIDENCE = (
    "risk_management_system",                    # Article 9
    "data_governance_record",                    # Article 10
    "technical_documentation",                   # Article 11
    "automatic_logging_design",                  # Article 12
    "instructions_for_use",                      # Article 13
    "human_oversight_plan",                      # Article 14
    "accuracy_robustness_cybersecurity_report",  # Article 15
    "quality_management_system",                 # Article 17
    "post_market_monitoring_plan",               # Article 72
)
ROUTE_EVIDENCE = {
    "annex_vi_internal_control": CORE_EVIDENCE + ("eu_database_registration",),
    "annex_vii_notified_body": CORE_EVIDENCE + (
        "eu_database_registration", "qms_approval_certificate",
        "technical_documentation_assessment_certificate"),
    "annex_i_product_procedure": CORE_EVIDENCE + ("integrated_product_conformity_assessment",),
    "not_high_risk": (),
    "none": (),
    "undetermined": (),
}
# Which evidence item holds the certificate an Annex V point 7 entry has to cite.
CERTIFICATE_ITEM = {
    "annex_vii_notified_body": "technical_documentation_assessment_certificate",
    "annex_i_product_procedure": "integrated_product_conformity_assessment",
}
RETENTION_YEARS = 10       # Article 47(1)

for _route in ("annex_vi_internal_control", "annex_vii_notified_body",
               "annex_i_product_procedure"):
    _extra = sorted(set(ROUTE_EVIDENCE[_route]) - set(CORE_EVIDENCE))
    print(f"{_route:28s} {len(ROUTE_EVIDENCE[_route]):>2d} items "
          f"= {len(CORE_EVIDENCE)} core + {len(_extra)} route-specific ({', '.join(_extra)})")
print(f"the notified-body route asks for "
      f"{len(ROUTE_EVIDENCE['annex_vii_notified_body']) - len(ROUTE_EVIDENCE['annex_vi_internal_control'])} "
      "items internal control does not")

In [ ]:
PROVIDER = {"name": "Northwind Analytics BV",
            "address": "Keizersgracht 1, 1015 CJ Amsterdam, Netherlands"}
ALL_EVIDENCE = tuple(sorted(set().union(*(set(v) for v in ROUTE_EVIDENCE.values()))))


def _pack(sid: str, name: str, kind: str, standards: tuple, placed: str,
          notified_body: dict | None = None, data_protection: str | None = None,
          overrides: dict | None = None) -> dict:
    """Build one evidence pack, fully populated, then apply the planted defects. Given to you."""
    prefix = "".join(w[0] for w in sid.split("-")).upper()
    evidence, artefacts = {}, {}
    for i, item in enumerate(ALL_EVIDENCE):
        art = f"ART-{sid}-{i:02d}"
        artefacts[art] = {"holds": item, "recorded_on": "2026-01-12"}
        evidence[item] = {"status": "present", "document": f"{prefix}-DOC-{i:02d}",
                          "date": "2026-01-12", "review_due": "2027-06-30", "source": art}
    for item, patch in (overrides or {}).items():
        evidence[item] = dict(evidence[item], **patch)
    return {"system_id": sid, "system_name": name, "system_type": kind,
            "provider": dict(PROVIDER), "standards_applied": tuple(standards),
            "data_protection": data_protection, "notified_body": notified_body,
            "placed_on_market": placed,
            "signatory": {"name": "R. Okonkwo", "function": "Head of Assurance",
                          "place": "Amsterdam"},
            "evidence": evidence, "artefacts": artefacts}


PACKS = {
    "face-match": _pack("face-match", "Face-match gate", "remote biometric identification",
                        ("EN 18228:2026",), "2026-03-02",
                        data_protection="Regulation (EU) 2016/679, DPIA FM-2025-11"),
    "border-biometrics": _pack(
        "border-biometrics", "Border categoriser", "biometric categorisation",
        ("EN 18228:2026 (in part)",), "2026-02-16",
        notified_body={"name": "Netherlands market surveillance authority (Article 74(8))",
                       "identification_number": "NL-MSA-0001"},
        data_protection="Regulation (EU) 2016/679, DPIA BB-2025-04"),
    # Four defects across the three buckets, including two different ways to be absent.
    "credit-scorer": _pack(
        "credit-scorer", "Credit scorer", "creditworthiness scoring", ("EN 18001:2026",),
        "2026-04-01",
        overrides={"human_oversight_plan": {"status": "planned", "document": ""},
                   # Status "present", and nothing behind it. P01-L01's checker knows.
                   "quality_management_system": {"document": "   "},
                   "data_governance_record": {"review_due": "2026-05-31"},
                   "instructions_for_use": {"source": "ART-withdrawn-2025"}}),
    "loan-copilot": _pack("loan-copilot", "Loan copilot", "creditworthiness scoring",
                          ("EN 18001:2026",), "2026-02-09"),
    "cv-ranker": _pack("cv-ranker", "CV ranker", "employment shortlisting",
                       ("EN 18001:2026",), "2026-06-01"),
    "lift-door-sensor": _pack(
        "lift-door-sensor", "Lift door sensor", "safety component of a lift", ("EN 81-20",),
        "2026-05-12",
        notified_body={"name": "Liftkeuringen Nederland", "identification_number": "0621"}),
    "mood-meter": _pack("mood-meter", "Mood meter", "emotion inference", (), "2026-03-01"),
    "shift-note-tidier": _pack("shift-note-tidier", "Shift-note tidier", "text reformatting",
                               (), "2026-03-15"),
    "legacy-sorter": _pack("legacy-sorter", "Legacy sorter", "application sorting", (),
                           "2026-01-05"),
}

SYSTEMS = {
    "face-match":        {"provided_digitally": False, "surface_permits_marking": True},
    "border-biometrics": {"provided_digitally": False, "surface_permits_marking": False},
    "credit-scorer":     {"provided_digitally": True, "machine_readable_access": True},
    # Digitally provided, with no way for anyone to reach the marking.
    "loan-copilot":      {"provided_digitally": True, "machine_readable_access": False},
    "cv-ranker":         {"provided_digitally": True, "machine_readable_access": True},
    "lift-door-sensor":  {"provided_digitally": False, "surface_permits_marking": True},
    "mood-meter":        {"provided_digitally": True, "machine_readable_access": True},
    "shift-note-tidier": {"provided_digitally": True, "machine_readable_access": True},
    "legacy-sorter":     {"provided_digitally": True, "machine_readable_access": False},
}

_cs = PACKS["credit-scorer"]["evidence"]
print(f"{len(PACKS)} packs, {len(ALL_EVIDENCE)} evidence items each")
_no_doc = sum(1 for r in _cs.values()
              if r["status"] != "present" or not str(r["document"]).strip())
_lapsed = sum(1 for r in _cs.values()
              if r["review_due"] and r["review_due"] < AS_OF.isoformat())
_lost = sum(1 for r in _cs.values()
            if r["source"] not in PACKS["credit-scorer"]["artefacts"])
print(f"credit-scorer's planted defects: {_no_doc} carrying no usable document "
      f"(one 'planned', one 'present' with a blank reference), {_lapsed} past review, "
      f"{_lost} pointing at an artefact the pack does not hold")

## 9. Exercise 6 — `readiness(route_record, pack, history, as_of)`

Three ways an evidence item fails, and they are not the same finding. **Missing** is a
procurement problem. **Out of date** is a maintenance problem. **Untraceable** — the item
is there, it names a document, and the artefact it claims to come from is not in the pack —
is the one that ends careers, because it looks green until somebody follows the pointer.

Each blocking item is reported in exactly one bucket, by the first test it fails. A stale
item is not also reported as untraceable; there is no point telling a reader to chase a
pointer on a document they have to reissue anyway.

<details><summary>💡 Hint 1 — what to think about</summary>

A status field is a claim; the document reference is the evidence. P01-L01's checker already
knows that a record marked present with a blank reference is absent, so ask it rather than
writing a second definition of present. The three buckets are exclusive and tested in order,
so a stale item is never also untraceable. And mind the ladder: a complete pack for a system
substantially modified since its assessment is not ready.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Build `required` from `ROUTE_EVIDENCE` for the route, run `completeness_report` over
`pack_as_trace(pack)`, and copy `present`, `missing` and `coverage` from it. For each PRESENT
item: out of date if it has no date, a date after `as_of`, or a `review_due` strictly before
`as_of`; otherwise untraceable if its source is not a key of `pack["artefacts"]`. Blocking is
the sorted union. Then walk the verdicts in the documented order — the two route-based ones,
then `route_reset`, then blocking — reading the pack and never writing to it.
</details>

In [ ]:
def readiness(route_record: dict, pack: dict, history: dict, as_of: date) -> dict:
    """Decide whether the evidence the chosen route requires is present, in date and traceable.

    Start by asking P01-L01's checker, not a new one:

        required = tuple(ROUTE_EVIDENCE[route_record["route"]])
        report = completeness_report(pack_as_trace(pack), required)

    Take "present", "missing" and "coverage" straight from that report. Then, over the
    PRESENT items only, and in this order, so every blocking item lands in exactly one bucket:

      out_of_date  the record fails any of: it has no "date"; its "date" is strictly AFTER
                   as_of; its "review_due" is set and is strictly BEFORE as_of. A review due
                   exactly on as_of has not lapsed.
      untraceable  it is not out_of_date, and its "source" is not a key of pack["artefacts"].
                   A missing or blank source is untraceable too.

    "blocking" is the sorted union of missing, out_of_date and untraceable.

    The verdict, in this order — the order is graded:
      "not_applicable"      route is "not_high_risk" or "none"
      "route_undetermined"  route is "undetermined"
      "reassessment_required" history["route_reset"] is truthy. BEFORE "blocked", because a
                             substantial modification the assessment has not seen means the
                             pack is complete for a system that no longer exists; filling the
                             gaps in it would be filling the gaps in the wrong pack.
      "blocked"             "blocking" is non-empty
      "ready"               otherwise

    Use parse_date() for every comparison, and do not mutate the pack.

    Example:
        >>> rr = assessment_route(RECORDS["mood-meter"], CONTEXTS["mood-meter"])
        >>> h = modification_history(CHANGE_LOGS["mood-meter"], PLANS["mood-meter"])
        >>> r = readiness(rr, PACKS["mood-meter"], h, AS_OF)
        >>> r["verdict"], r["coverage"]
        ('not_applicable', 1.0)

    Returns:
        dict with exactly these twelve keys:
          "system_id"    route_record["system_id"]
          "route"        route_record["route"]
          "required"     list[str], sorted
          "present"      list[str], sorted       (from completeness_report)
          "missing"      list[str], sorted       (from completeness_report)
          "out_of_date"  list[str], sorted
          "untraceable"  list[str], sorted
          "coverage"     float                   (from completeness_report)
          "blocking"     list[str], sorted
          "route_reset"  bool, bool(history["route_reset"])
          "verdict"      str
          "ready"        bool, True only when verdict == "ready"
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_readiness() -> None:
    def run(sid):
        rr = assessment_route(RECORDS[sid], CONTEXTS[sid])
        hist = modification_history(CHANGE_LOGS[sid], PLANS[sid])
        return readiness(rr, PACKS[sid], hist, AS_OF)

    got = run("face-match")
    assert set(got) == {"system_id", "route", "required", "present", "missing", "out_of_date",
                        "untraceable", "coverage", "blocking", "route_reset", "verdict",
                        "ready"}, (
        f"keys were {sorted(got)} — return exactly the twelve documented names.")
    assert got["verdict"] == "ready" and got["ready"] is True, (
        f"face-match's pack is complete, in date and traceable, and nothing has been "
        f"modified since its assessment; got {got['verdict']!r} with "
        f"blocking={got['blocking']}.")
    assert got["coverage"] == 1.0 and len(got["required"]) == len(
        ROUTE_EVIDENCE["annex_vi_internal_control"]), (
        "coverage and the required list come from completeness_report on this route's item "
        "list — not from the whole pack.")

    credit = run("credit-scorer")
    assert credit["verdict"] == "blocked", (
        f"credit-scorer has four planted defects; got {credit['verdict']!r}.")
    assert credit["missing"] == ["human_oversight_plan", "quality_management_system"], (
        f"missing came back {credit['missing']} — a 'planned' record and a 'present' record "
        "whose document reference is three spaces are BOTH absent to P01-L01's checker. A "
        "status field is a claim; the document reference is the evidence.")
    assert credit["out_of_date"] == ["data_governance_record"], (
        f"out_of_date came back {credit['out_of_date']} — that record's review_due is "
        "2026-05-31, which is behind us.")
    assert credit["untraceable"] == ["instructions_for_use"], (
        f"untraceable came back {credit['untraceable']} — it names ART-withdrawn-2025 and the "
        "pack's artefact index has no such key.")
    assert credit["blocking"] == sorted(["human_oversight_plan", "quality_management_system",
                                         "data_governance_record", "instructions_for_use"]), (
        f"blocking is the sorted union of the three buckets; got {credit['blocking']}.")

    for sid, expected in (("mood-meter", "not_applicable"),
                          ("shift-note-tidier", "not_applicable"),
                          ("legacy-sorter", "route_undetermined"),
                          ("border-biometrics", "ready"), ("loan-copilot", "ready"),
                          ("lift-door-sensor", "ready")):
        verdict = run(sid)["verdict"]
        assert verdict == expected, f"{sid} should be {expected!r}, got {verdict!r}."

    cv = run("cv-ranker")
    assert cv["blocking"] == [] and cv["coverage"] == 1.0, (
        "cv-ranker's pack is immaculate — that is the point of it.")
    assert cv["verdict"] == "reassessment_required" and cv["ready"] is False, (
        f"cv-ranker came back {cv['verdict']!r}. Its pack is complete, and CHG-63 modified the "
        "intended purpose after the assessment on file, so the route resets. A complete pack "
        "for a superseded assessment is not readiness.")
    assert run("mood-meter")["coverage"] == 1.0, (
        "a route with no required items is vacuously covered, not a division by zero — "
        "that is what P01-L01's checker already returns.")
    print("exercise 6 looks right")


_try("exercise 6", _check_readiness)

## 10. Exercise 7 — `declaration_of_conformity(...)`

Article 47 asks the provider to draw up a written declaration for each high-risk system and
keep it at the disposal of the national competent authorities for ten years after the
system is placed on the market. Annex V says what goes in it. By drawing it up, the
provider assumes responsibility for compliance.

That last sentence is why this function's most important behaviour is refusing. A
declaration is not a document you produce and then check; it is the act of taking
responsibility, and there is no such thing as taking responsibility provisionally.

<details><summary>💡 Hint 1 — what to think about</summary>

The most important thing this function does is refuse. Which single field of the readiness
report decides that, and why is passing it through as the reason more useful than a generic
"not ready"? When it does issue, two fields are easy to get wrong: the retention clock starts
at a date in the pack, not today; and point 7 follows whether the ROUTE needed a body, not
whether the pack happens to name one.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Return the refusal shape at once for any verdict but ready. Otherwise assemble the nine keys
from the docstring: copy the provider dict rather than sharing it, sort the standards, and
sign with the signatory's place, name and function and the `as_of` date. For point 7 use None
unless `notified_body_required`; then copy the body's name and number from the pack and the
certificate from the evidence item `CERTIFICATE_ITEM` names for this route. `keep_until` is
`plus_years` of the parsed placing date. Seal last, so the hash covers everything else.
</details>

In [ ]:
def declaration_of_conformity(route_record: dict, readiness_report: dict, pack: dict,
                              as_of: date) -> dict:
    """Emit the Annex V declaration, or refuse and say which verdict stopped it.

    REFUSE unless readiness_report["verdict"] == "ready". On a refusal return
    {"issued": False, "refusal_reason": readiness_report["verdict"], "declaration": None} —
    the verdict IS the reason, so a reader gets "reassessment_required" rather than a generic
    "not ready".

    Otherwise build the declaration with exactly these nine keys and seal it with seal():

      "annex_v_1_system"    f"{pack['system_name']} · {pack['system_type']} · {system_id}"
                            — name, type and an unambiguous reference, per Annex V point 1
      "annex_v_2_provider"  a COPY of pack["provider"]  (point 2)
      "annex_v_3_sole_responsibility"  True             (point 3)
      "annex_v_4_conformity"  f"in conformity with Regulation (EU) 2024/1689 via "
                              f"{ROUTE_LABEL[route]}"   (point 4)
      "annex_v_5_data_protection"  pack["data_protection"], which may be None  (point 5)
      "annex_v_6_standards"   sorted(pack["standards_applied"])  (point 6)
      "annex_v_7_notified_body"  (point 7) None when route_record["notified_body_required"]
                            is falsy. Otherwise a dict with exactly "name",
                            "identification_number" and "certificate", the first two copied
                            from pack["notified_body"] and the third being the "document" of
                            pack["evidence"][CERTIFICATE_ITEM[route]].
      "annex_v_8_signature"   (point 8) a dict with exactly "place", "date", "name" and
                            "function": place/name/function from pack["signatory"] and date
                            from as_of.isoformat().
      "keep_until"          plus_years(parse_date(pack["placed_on_market"]),
                            RETENTION_YEARS).isoformat() — Article 47(1) runs from the
                            PLACING, not from today.

    seal() adds the tenth key, "record_hash".

    Example:
        >>> res = declaration_of_conformity(
        ...     {"system_id": "x", "route": "none", "notified_body_required": False},
        ...     {"verdict": "not_applicable"}, PACKS["mood-meter"], AS_OF)
        >>> res["issued"], res["refusal_reason"], res["declaration"]
        (False, 'not_applicable', None)

    Returns:
        dict with exactly "issued" (bool), "refusal_reason" (str | None) and
        "declaration" (dict | None).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_declaration() -> None:
    def run(sid):
        rr = assessment_route(RECORDS[sid], CONTEXTS[sid])
        hist = modification_history(CHANGE_LOGS[sid], PLANS[sid])
        ready = readiness(rr, PACKS[sid], hist, AS_OF)
        return rr, declaration_of_conformity(rr, ready, PACKS[sid], AS_OF)

    _, res = run("face-match")
    assert set(res) == {"issued", "refusal_reason", "declaration"}, (
        f"keys were {sorted(res)} — return exactly the three documented names.")
    assert res["issued"] is True and res["refusal_reason"] is None, (
        "face-match is ready, so a declaration is due.")
    decl = res["declaration"]
    assert set(decl) == {"annex_v_1_system", "annex_v_2_provider",
                         "annex_v_3_sole_responsibility", "annex_v_4_conformity",
                         "annex_v_5_data_protection", "annex_v_6_standards",
                         "annex_v_7_notified_body", "annex_v_8_signature", "keep_until",
                         "record_hash"}, (
        f"declaration keys were {sorted(decl)} — Annex V's eight points, keep_until, and the "
        "seal.")
    assert decl["annex_v_7_notified_body"] is None, (
        "face-match travels internal control; naming a notified body it never used is a false "
        "statement in a document whose whole purpose is assuming responsibility.")
    assert decl["keep_until"] == "2036-03-02", (
        f"keep_until came back {decl['keep_until']!r} — Article 47(1)'s ten years run from "
        "placed_on_market (2026-03-02), not from as_of.")
    assert seal(decl)["record_hash"] == decl["record_hash"], (
        "the declaration must reseal to the same digest — compute it over the record WITHOUT "
        "record_hash in it, as seal() does.")

    rr, res = run("border-biometrics")
    body = res["declaration"]["annex_v_7_notified_body"]
    assert set(body) == {"name", "identification_number", "certificate"}, (
        f"annex_v_7 keys were {sorted(body)} — exactly name, identification_number and "
        "certificate.")
    assert body["certificate"] == PACKS["border-biometrics"]["evidence"][
        "technical_documentation_assessment_certificate"]["document"], (
        "the certificate cited is the Annex VII technical documentation assessment "
        "certificate in the pack, not the QMS approval and not a made-up reference.")

    _, res = run("lift-door-sensor")
    assert res["declaration"]["annex_v_7_notified_body"] is None, (
        "lift-door-sensor's context does not say a notified body was involved, so Annex V "
        "point 7 is empty — read notified_body_required, not pack['notified_body'].")

    for sid, reason in (("cv-ranker", "reassessment_required"),
                        ("credit-scorer", "blocked"),
                        ("mood-meter", "not_applicable"),
                        ("shift-note-tidier", "not_applicable"),
                        ("legacy-sorter", "route_undetermined")):
        _, res = run(sid)
        assert res["issued"] is False and res["declaration"] is None, (
            f"{sid} is not ready, so no declaration may be drawn up; you issued one.")
        assert res["refusal_reason"] == reason, (
            f"{sid} refused with {res['refusal_reason']!r}, expected {reason!r} — the "
            "readiness verdict is the reason.")
    print("exercise 7 looks right")


_try("exercise 7", _check_declaration)

## 11. Exercise 8 — `ce_marking_record(route_record, declaration_result, system)`

Article 48 is short and every sentence of it is a branch. The marking is affixed visibly,
legibly and indelibly; where that is not possible or not warranted, it goes on the
packaging or the accompanying documentation. For a system provided digitally there is a
digital CE marking — **only if** it can easily be accessed through the interface or a
machine-readable code. And where applicable the marking is followed by the identification
number of the notified body responsible for the Article 43 procedure.

The system in this lesson that fails here fails on the word "only". Its declaration is
valid and its pack is clean, and there is still no lawful way to mark it.

<details><summary>💡 Hint 1 — what to think about</summary>

Article 48 is a decision tree, and its order is the trap: nothing about the system matters
until you know there is a declaration to attest to. Then read the word "only" in 48(2): a
digitally provided system without an accessible marking does not fall back to another form.
And where does the notified body's number come from — the pack, which may name any body, or
the declaration, which names the one responsible for this route?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Refuse on a missing declaration first, then on digital-without-access. Otherwise the form is
digital if the system is provided digitally, physical if its surface permits marking, and the
accompanying documentation otherwise. The number is the `identification_number` inside the
declaration's point 7 entry, or None when that entry is None; the hash is the declaration's
`record_hash`; and the basis depends only on whether the form is digital.
</details>

In [ ]:
def ce_marking_record(route_record: dict, declaration_result: dict, system: dict) -> dict:
    """Emit the Article 48 CE marking record, or refuse.

      1. declaration_result["issued"] is falsy
         -> refuse with "no_declaration_of_conformity". The marking attests to a declaration;
            without one there is nothing to attest to.
      2. system["provided_digitally"] truthy and system["machine_readable_access"] falsy
         -> refuse with "digital_marking_not_accessible" (Article 48(2)'s "only if").
      3. otherwise the form is:
            "digital"                    system["provided_digitally"] truthy
            "physical"                   otherwise, if system["surface_permits_marking"] truthy
            "accompanying_documentation" otherwise
         and the record is affixed.

    On a refusal return {"affixed": False, "refusal_reason": <id>, "marking": None}.

    On success "marking" is a dict with exactly these four keys:
      "form"                 as above
      "notified_body_number" the "identification_number" from the declaration's
                             "annex_v_7_notified_body", or None when that entry is None.
                             Take it from the DECLARATION: Article 48(4) names the body
                             responsible for the Article 43 procedure, which is the body the
                             declaration already cites.
      "declaration_hash"     the declaration's "record_hash"
      "basis"                "Article 48(2)" when the form is "digital", "Article 48(3)"
                             otherwise

    Example:
        >>> ce_marking_record({}, {"issued": False}, {})["refusal_reason"]
        'no_declaration_of_conformity'

    Returns:
        dict with exactly "affixed" (bool), "refusal_reason" (str | None) and
        "marking" (dict | None).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_ce_marking() -> None:
    def run(sid):
        rr = assessment_route(RECORDS[sid], CONTEXTS[sid])
        hist = modification_history(CHANGE_LOGS[sid], PLANS[sid])
        ready = readiness(rr, PACKS[sid], hist, AS_OF)
        decl = declaration_of_conformity(rr, ready, PACKS[sid], AS_OF)
        return decl, ce_marking_record(rr, decl, SYSTEMS[sid])

    decl, got = run("face-match")
    assert set(got) == {"affixed", "refusal_reason", "marking"}, (
        f"keys were {sorted(got)} — return exactly the three documented names.")
    assert got["affixed"] is True and got["marking"]["form"] == "physical", (
        f"face-match is not provided digitally and its surface takes a marking, so the form "
        f"is 'physical'; got {got['marking']['form']!r}.")
    assert got["marking"]["notified_body_number"] is None, (
        "no notified body was involved, so Article 48(4) adds no number — 'where applicable' "
        "is doing the work in that sentence.")
    assert got["marking"]["declaration_hash"] == decl["declaration"]["record_hash"], (
        "the marking cites the declaration it attests to, by its seal.")

    _, got = run("border-biometrics")
    assert got["marking"]["form"] == "accompanying_documentation", (
        f"border-biometrics' surface does not take a marking, so Article 48(3) sends it to "
        f"the accompanying documentation; got {got['marking']['form']!r}.")
    assert got["marking"]["notified_body_number"] == "NL-MSA-0001", (
        f"got {got['marking']['notified_body_number']!r} — read it off the declaration's "
        "Annex V point 7, which is the body responsible for the Article 43 procedure.")

    _, got = run("loan-copilot")
    assert got["affixed"] is False and got["refusal_reason"] == "digital_marking_not_accessible", (
        f"loan-copilot's declaration is valid and its pack is clean; got "
        f"{got['refusal_reason']!r}. Article 48(2) permits a digital marking ONLY if it can "
        "easily be accessed, and this one cannot.")
    _, got = run("cv-ranker")
    assert got["affixed"] is False and got["refusal_reason"] == "no_declaration_of_conformity", (
        "cv-ranker has no declaration, so there is nothing for a marking to attest to.")
    _, got = run("credit-scorer")
    assert got["marking"] is None, "a refused marking carries no marking record."
    print("exercise 8 looks right")


_try("exercise 8", _check_ce_marking)

## 12. The artefact

One record per system: the route, the readiness verdict, the declaration if there is one,
the CE marking if there is one, and the modification history that justifies all of it —
sealed with the same digest P01-L02 used, so it appends to your P01-L01 log unchanged.

In [ ]:
def conformity_assessment_record(system_id: str, as_of: date = AS_OF) -> dict:
    """Compose the whole chain for one system and seal it. Given to you; not graded."""
    route = assessment_route(RECORDS[system_id], CONTEXTS[system_id])
    history = modification_history(CHANGE_LOGS[system_id], PLANS[system_id])
    ready = readiness(route, PACKS[system_id], history, as_of)
    declaration = declaration_of_conformity(route, ready, PACKS[system_id], as_of)
    marking = ce_marking_record(route, declaration, SYSTEMS[system_id])
    return seal({
        "system_id": system_id,
        "as_of": as_of.isoformat(),
        "classification_record_hash": RECORDS[system_id]["record_hash"],
        "route": route,
        "modification_history": history,
        "readiness": ready,
        "eu_declaration_of_conformity": declaration,
        "ce_marking_record": marking,
    })


def _show_pack_table() -> None:
    print(f"{'system':20s} {'route':26s} {'verdict':22s} {'decl':>5s}  marking")
    issued = marked = 0
    for sid in RECORDS:
        rec = conformity_assessment_record(sid)
        decl, ce = rec["eu_declaration_of_conformity"], rec["ce_marking_record"]
        issued += bool(decl["issued"])
        marked += bool(ce["affixed"])
        mark = ce["marking"]["form"] if ce["affixed"] else f"refused: {ce['refusal_reason']}"
        print(f"  {sid:18s} {rec['route']['route']:26s} {rec['readiness']['verdict']:22s} "
              f"{'yes' if decl['issued'] else ' no':>5s}  {mark}")
    print(f"\n{len(RECORDS)} systems, {issued} declarations, {marked} CE markings. "
          f"{issued - marked} system(s) earned a declaration and still cannot be marked.")


_try("the pack", _show_pack_table, needs=tuple(_EXERCISES))

In [ ]:
def _show_the_trap() -> None:
    """cv-ranker, in the order an inspector would meet it."""
    rec = conformity_assessment_record("cv-ranker")
    ready, hist = rec["readiness"], rec["modification_history"]
    print(f"cv-ranker evidence: {len(ready['present'])}/{len(ready['required'])} present, "
          f"coverage {ready['coverage']:.2f}, missing {ready['missing']}, "
          f"out of date {ready['out_of_date']}, untraceable {ready['untraceable']}")
    print(f"blocking items: {len(ready['blocking'])}")
    for verdict in hist["assessed"]:
        flag = "SUBSTANTIAL" if verdict["substantial"] else "           "
        print(f"  {verdict['on']}  {verdict['change_id']:8s} {flag}  {verdict['reason']}")
    print(f"assessment on file: {PLANS['cv-ranker']['assessed_on']} · "
          f"uncovered substantial changes: {hist['since_assessment']}")
    print(f"verdict: {ready['verdict']} · declaration issued: "
          f"{rec['eu_declaration_of_conformity']['issued']}")
    print("A pack with nothing missing, and no declaration. That is the whole lesson.")


_try("the trap", _show_the_trap, needs=tuple(_EXERCISES))

## 13. Common mistakes

* **Running the Article 43(1) standards test on a points 2-to-8 system.** It is the single
  most common wrong answer, because 43(1) is the longer and more interesting paragraph.
  43(2) is unconditional: internal control, no notified body, no standards test.
* **Defaulting an unreadable Annex III point to something.** A record whose point cannot be
  read is `undetermined`, which blocks. Defaulting it to internal control is choosing the
  cheaper route on the strength of a typo.
* **Treating "pre-determined" as the whole of Article 43(4).** The carve-out is
  pre-determined *and* in the Annex IV point 2(f) description. One without the other is a
  substantial modification.
* **Resetting the route on every substantial modification.** The ones the assessment on
  file already covers are covered. A history that cannot distinguish them tells a provider
  who re-assessed that they are in the same position as one who did not.
* **Collapsing missing, stale and untraceable into "not ready".** Three different people
  fix those three findings.
* **Checking readiness after drawing up the declaration.** Article 47(4) makes drawing it
  up the act of assuming responsibility. There is no draft state.
* **Reading the notified body's number off the pack rather than the declaration.** The pack
  may name a body that had no part in this route. Article 48(4) names the body responsible
  for the Article 43 procedure, and that is the one Annex V point 7 already cites.

## 14. Self-check

1. A creditworthiness system falls in Annex III point 5(b). Its provider applied no
   harmonised standard at all, and no common specification exists. Which route?
   - (a) Annex VII, because the standards gap forces a notified body
   - (b) undetermined, until a standard is published
   - (c) Annex VI internal control — Article 43(2) does not consult the standards
   - (d) the product legislation's own procedure

2. A provider pre-determined a quarterly retrain and never wrote it into the Annex IV
   point 2(f) description. The retrain affects Chapter III Section 2 compliance. It is:
   - (a) not substantial; it was pre-determined
   - (b) substantial; the carve-out needs the documentation as well as the intention
   - (c) not substantial; retraining is never substantial
   - (d) substantial only if the intended purpose also changed

In [ ]:
# Before questions 3 to 5: two facts you can run rather than recall.
def _self_check_aids() -> None:
    cv = conformity_assessment_record("cv-ranker")
    loan = conformity_assessment_record("loan-copilot")
    print(f"cv-ranker   blocking={len(cv['readiness']['blocking'])} "
          f"reset={cv['readiness']['route_reset']} "
          f"verdict={cv['readiness']['verdict']}")
    print(f"loan-copilot ready={loan['readiness']['ready']} "
          f"declaration={loan['eu_declaration_of_conformity']['issued']} "
          f"marking={loan['ce_marking_record']['refusal_reason']}")
    credit = conformity_assessment_record("credit-scorer")["readiness"]
    print(f"credit-scorer missing={credit['missing']} out_of_date={credit['out_of_date']} "
          f"untraceable={credit['untraceable']}")


_try("self-check aids", _self_check_aids, needs=tuple(_EXERCISES))

3. A readiness report comes back with coverage 1.00 and nothing missing, stale or
   untraceable, and a substantial modification dated after the assessment on file. The
   declaration should be:
   - (a) issued; the evidence is all there
   - (b) issued with a caveat naming the modification
   - (c) issued, and withdrawn if an authority objects
   - (d) not issued; the pack is complete for an assessment that no longer covers the system

4. Which of these makes an evidence item **untraceable** rather than **missing**?
   - (a) its status is "planned"
   - (b) it is present and referenced, and the artefact its source names is not in the pack
   - (c) its review date has passed
   - (d) it was issued in the future

5. A digitally provided system has a valid declaration and no machine-readable route to its
   CE marking. What follows?
   - (a) affix a physical marking instead
   - (b) affix the digital marking; accessibility is a deployer problem
   - (c) no marking may be affixed — Article 48(2) permits the digital form only if it can
         easily be accessed
   - (d) the declaration is invalid too

Mark them in the next cell. The key is not written in this file — only a salted hash of it —
so you find out which are wrong without reading the answers off the page.

In [ ]:
# Salted hashes of the answers, not the answers. Nothing here tells you which letter is right.
_SELF_CHECK_KEY = {
    1: "ed62bfeca1bb6e6a",
    2: "4927b091d401b5f9",
    3: "918b0ad225e700d6",
    4: "d3c91663c385e12a",
    5: "9659038c838f691e",
}

_SELF_CHECK_HINT = {
    1: "re-read the asymmetry note in section 1, then look at what _check_assessment_route "
       "asserts about credit-scorer.",
    2: "re-read rule 3 of the exercise 4 docstring, and count the halves.",
    3: "run the section 12 trap cell and read the last two lines.",
    4: "look at which bucket credit-scorer's instructions_for_use lands in, and why.",
    5: "run the self-check aids cell above and read the loan-copilot line.",
}


def check_self_check(answers: dict) -> None:
    """Mark your self-check answers. Pass a dict of question number -> letter.

    Example:
        >>> check_self_check({1: "a"})          # doctest: +SKIP
          q1  not 'a' — re-read the asymmetry note in section 1 ...
          q2  no answer given
        ...
    """
    right = 0
    for question in sorted(_SELF_CHECK_KEY):
        given = str(answers.get(question, "")).strip().lower()
        digest = hashlib.sha256(f"P01-L08:q{question}:{given}".encode()).hexdigest()[:16]
        if digest == _SELF_CHECK_KEY[question]:
            right += 1
            print(f"  q{question}  correct")
        elif not given:
            print(f"  q{question}  no answer given")
        else:
            print(f"  q{question}  not {given!r} — {_SELF_CHECK_HINT[question]}")
    print(f"\n{len(_SELF_CHECK_KEY)} questions, {right} right")


# Put your own letters in, then run this cell:
# check_self_check({1: "a", 2: "a", 3: "a", 4: "a", 5: "a"})

## What you built, and where it goes next

A route decision that is a decision procedure rather than a judgement call, a readiness
verdict that names three different kinds of failure, a declaration that cannot be drawn up
while the pack is incomplete, and a CE marking that cannot be affixed without a
declaration. Each of those is one of the evidence ids the conformity pack from
`T10-L01-ai-act-conformity-pack` has been holding open: `conformity_assessment_record`,
`eu_declaration_of_conformity`, `ce_marking_record`.

The chain runs both ways. The record you sealed here carries the hash of the P01-L02
classification record it was derived from, so an inspector who does not believe the route
can walk back to the question that produced the tier. Append it to your P01-L01 log and the
whole derivation is tamper-evident.

**Again, and finally: this is engineering, not legal advice.**

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_annex_iii_point),
                              ("exercise 2", _check_standards_gap),
                              ("exercise 3", _check_assessment_route),
                              ("exercise 4", _check_is_substantial),
                              ("exercise 5", _check_modification_history),
                              ("exercise 6", _check_readiness),
                              ("exercise 7", _check_declaration),
                              ("exercise 8", _check_ce_marking)):
            _try(_name, _check)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))